
# Табличный редактор `df_out`

Интерактивный редактор на `ipywidgets` с пагинацией.

Возможности:

- поиск по УНП, клиенту и номеру договора;
- фильтры по валюте и типу операции;
- выбор отчетной даты;
- 10 / 25 / 50 строк на странице;
- редактирование сразу нескольких строк;
- редактирование `задолженность_{дата}`;
- автоматический пересчет `OD_{дата}` по валютному курсу из `df_rates`;
- редактирование НИ, ПФН, НВВ, рестры, обеспеченности, ГР и % резервирования;
- сохранение всей текущей страницы обратно в `df_out`;
- журнал ручных изменений `manual_edit_log`.

**Перед запуском должны существовать `df_out` и `df_rates`.**

Ожидаемая структура `df_rates`: валюты по строкам, отчетные даты по столбцам.  
Валюта может быть уже индексом либо находиться в обычном столбце `валюта` / `Валюта`.


In [ ]:

import re
import math
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output


## 1. Настройки и подготовка курсов

In [ ]:

# =============================================================================
# НАЗВАНИЯ ПОСТОЯННЫХ СТОЛБЦОВ В df_out
# Если у тебя они называются иначе — поменяй только эти значения.
# =============================================================================

COL_UNN = "УНП"
COL_CLIENT = "Наименование клиента"
COL_CONTRACT = "Номер договора"
COL_CURRENCY = "Валюта"
COL_OPERATION = "Тип операции"


# =============================================================================
# ПРОВЕРКИ
# =============================================================================

if "df_out" not in globals():
    raise NameError("Сначала должен быть создан dataframe df_out")

if "df_rates" not in globals():
    raise NameError("Сначала должен быть создан dataframe df_rates")

required_static_columns = [
    COL_UNN,
    COL_CLIENT,
    COL_CONTRACT,
    COL_CURRENCY,
    COL_OPERATION,
]

missing_static_columns = [
    col for col in required_static_columns
    if col not in df_out.columns
]

if missing_static_columns:
    raise ValueError(
        "В df_out отсутствуют обязательные столбцы: "
        + ", ".join(missing_static_columns)
    )


# =============================================================================
# ОТЧЕТНЫЕ ДАТЫ ИЗ САМОГО df_out
# =============================================================================

report_dates = []

for col in df_out.columns:
    match = re.match(
        r"^задолженность_(\d{2}\.\d{2}\.\d{4})$",
        str(col)
    )

    if match:
        report_dates.append(match.group(1))

report_dates = sorted(
    set(report_dates),
    key=lambda x: pd.to_datetime(x, format="%d.%m.%Y")
)

if not report_dates:
    raise ValueError(
        "В df_out не найдены столбцы вида "
        "'задолженность_01.01.2026'"
    )


# =============================================================================
# ФАКТОРЫ, КОТОРЫЕ ДЕЙСТВУЮТ ВПЕРЕД ПО ВРЕМЕНИ
# =============================================================================

FACTOR_NAMES = [
    "НИ",
    "ПФН",
    "НВВ",
    "рестра",
    "обеспеченность",
    "ГР",
    "%рез",
]


# =============================================================================
# ПЕРВОНАЧАЛЬНОЕ ПРОТЯГИВАНИЕ ПЕРВОЙ ОТЧЕТНОЙ ДАТЫ
# =============================================================================
#
# На момент запуска редактора считаем, что факторы первой отчетной даты
# являются исходным состоянием. Поэтому один раз переносим их на все
# последующие отчетные даты.
#
# ВАЖНО:
# - задолженность и OD здесь НЕ протягиваются;
# - только НИ, ПФН, НВВ, рестра, обеспеченность, ГР и %рез;
# - внутри одного kernel повторный запуск этой ячейки не перезапишет
#   уже сделанные через виджет изменения.
# =============================================================================

_factor_init_signature = (
    id(df_out),
    tuple(report_dates),
)

if globals().get("_factor_init_signature_done") != _factor_init_signature:

    first_report_date = report_dates[0]

    for factor in FACTOR_NAMES:

        source_col = f"{factor}_{first_report_date}"

        if source_col not in df_out.columns:
            continue

        for future_date in report_dates[1:]:

            target_col = f"{factor}_{future_date}"

            if target_col not in df_out.columns:
                continue

            # Копируем состояние первой даты на следующую дату.
            df_out[target_col] = df_out[source_col].copy()

    _factor_init_signature_done = _factor_init_signature

    print(
        f"Факторы первой отчетной даты {first_report_date} "
        f"протянуты на все последующие даты"
    )


# =============================================================================
# ПОДГОТОВКА КОПИИ ТАБЛИЦЫ КУРСОВ
# Оригинальный df_rates не меняем.
# =============================================================================

rates_table = df_rates.copy()

currency_col_in_rates = None

for col in rates_table.columns:
    if str(col).strip().lower() == "валюта":
        currency_col_in_rates = col
        break

if currency_col_in_rates is not None:
    rates_table[currency_col_in_rates] = (
        rates_table[currency_col_in_rates]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    rates_table = rates_table.set_index(
        currency_col_in_rates
    )

rates_table.index = (
    rates_table.index
    .astype(str)
    .str.strip()
    .str.upper()
)


# Приводим только те названия столбцов, которые можно распознать как даты.
rename_rates_columns = {}

for col in rates_table.columns:
    try:
        parsed = pd.to_datetime(
            col,
            dayfirst=True,
            errors="raise"
        )

        rename_rates_columns[col] = parsed.strftime(
            "%d.%m.%Y"
        )

    except Exception:
        pass

rates_table = rates_table.rename(
    columns=rename_rates_columns
)


def get_fx_rate(currency, date):
    currency = str(currency).strip().upper()

    if currency not in rates_table.index:
        return np.nan

    if date not in rates_table.columns:
        return np.nan

    value = rates_table.at[
        currency,
        date
    ]

    try:
        return float(value)
    except Exception:
        return np.nan


print("Отчетные даты:", report_dates)
print("Валют в таблице курсов:", len(rates_table.index))


## 2. Автоматический расчет расходов на резервы

После заполнения факторов по всем отчетным датам рассчитываются три компонента:

1. **Изменение качества**

   `-OD текущего периода × (%рез текущий - %рез прошлый) / 100`

   Эта же сумма раскладывается по факторным столбцам НИ / ПФН / НВВ / рестра / обеспеченность.

2. **Переоценка**

   `задолженность прошлого периода × (курс текущего периода - курс прошлого периода) × %рез прошлый / 100`

3. **Изменение портфеля**

   `-(OD текущий × %рез текущий / 100 - OD прошлый × %рез прошлый / 100) - переоценка`

Первая отчетная дата получает нули, так как предыдущего периода для нее нет. После сохранения изменений через `ipywidgets` весь расчет выполняется автоматически заново.


In [ ]:

import pandas as pd
import numpy as np
import re


# =============================================================================
# НАСТРОЙКИ
# =============================================================================

# Названия уже заполненных столбцов внутри каждого отчетного периода.
SOURCE_COLUMNS = {
    "debt": "задолженность",
    "od": "OD",
    "ni": "НИ",
    "pfn": "ПФН",
    "nvv": "НВВ",
    "restra": "рестра",
    "security": "обеспеченность",
    "group": "ГР",
    "rate": "%рез",
}

# Постоянный столбец с типом позиции.
BALANCE_COLUMN = "Баланс/внебаланс"

# Допустимая техническая погрешность при сравнении чисел.
EPSILON = 1e-7


# Факторные столбцы.
# Их сумма должна быть равна столбцу "ухудшение качества_{дата}".
FACTOR_COLUMNS = [
    "ушла НИ",
    "пришла НИ",
    "ушел ПФН",
    "пришел ПФН",
    "ушло НВВ",
    "пришло НВВ",
    "изменение рестры",
    "изменилась обеспеченность",
]


# Все расчетные столбцы, которые ноутбук создаст самостоятельно.
CALC_COLUMNS = FACTOR_COLUMNS + [
    "ухудшение качества",
    "проверка",
    "переоценка",
    "изменение портфеля",
]



def make_col(name, period):
    """Формирует имя столбца вида 'НИ_01.02.2026'."""
    return f"{name}_{period}"


def find_periods(df):
    """Находит все отчетные даты по названиям столбцов задолженности."""

    pattern = re.compile(
        rf"^{re.escape(SOURCE_COLUMNS['debt'])}_(\d{{2}}\.\d{{2}}\.\d{{4}})$"
    )

    periods = []

    for column in df.columns:
        match = pattern.match(str(column))

        if match:
            periods.append(match.group(1))

    # Убираем дубли и сортируем как даты, а не как строки.
    periods = sorted(
        set(periods),
        key=lambda x: pd.to_datetime(x, format="%d.%m.%Y"),
    )

    return periods


def to_float(value):
    """
    Безопасно преобразует значение в float.

    Поддерживает:
    5
    "5"
    "5,0"
    " 5 "

    Если преобразование невозможно — возвращает np.nan.
    np.nan здесь используется только как технический внутренний маркер;
    в итоговые расчетные столбцы он не записывается.
    """

    if pd.isna(value):
        return np.nan

    try:
        value = (
            str(value)
            .strip()
            .replace("\xa0", "")
            .replace(" ", "")
            .replace(",", ".")
        )

        return float(value)

    except (ValueError, TypeError):
        return np.nan


def normalize_scalar(value):
    """Нормализует текст для устойчивого сравнения."""

    if pd.isna(value):
        return None

    return (
        str(value)
        .strip()
        .lower()
        .replace("ё", "е")
    )


def is_active(value):
    """
    Переводит НИ / ПФН / НВВ в True / False.

    Основной ожидаемый формат:
    0 -> признака нет
    1 -> признак есть

    Дополнительно понимает 'да', '+', True, 'есть'.
    """

    if pd.isna(value):
        return False

    numeric = to_float(value)

    if not pd.isna(numeric):
        return numeric == 1

    text = normalize_scalar(value)

    return text in {
        "да",
        "true",
        "есть",
        "yes",
        "+",
    }


def contains_restra(value):
    """Определяет наличие реструктуризации / рестры."""

    if pd.isna(value):
        return False

    if is_active(value):
        return True

    text = normalize_scalar(value)

    if text is None:
        return False

    return (
        "рестр" in text
        or
        "реестр" in text
    )


def same_group(first_group, second_group):
    """Сравнивает группы риска: 2, 2.0 и '2' считаются одной ГР."""

    first_numeric = to_float(first_group)
    second_numeric = to_float(second_group)

    if (
        not pd.isna(first_numeric)
        and
        not pd.isna(second_numeric)
    ):
        return first_numeric == second_numeric

    return (
        normalize_scalar(first_group)
        ==
        normalize_scalar(second_group)
    )



# =============================================================================
# СПРАВОЧНИК СТАВОК ПРОМЕЖУТОЧНЫХ ГР
# =============================================================================

def rebuild_rate_lookup():
    """
    Перестраивает справочник %рез по текущему состоянию df_out.

    Это выполняется перед каждым пересчетом, поэтому ручные изменения ГР/%рез
    через виджет сразу учитываются в факторном анализе.
    """
    global rate_lookup, rate_lookup_conflicts

    rate_lookup = {}
    conflicts = []

    periods = find_periods(df_out)

    for period in periods:
        group_column = make_col(SOURCE_COLUMNS["group"], period)
        rate_column = make_col(SOURCE_COLUMNS["rate"], period)

        for pos in range(len(df_out)):
            row = df_out.iloc[pos]

            balance_type = normalize_scalar(
                row[BALANCE_COLUMN]
            )

            risk_group = to_float(
                row[group_column]
            )

            reserve_rate = to_float(
                row[rate_column]
            )

            if (
                balance_type is None
                or pd.isna(risk_group)
                or pd.isna(reserve_rate)
            ):
                continue

            key = (
                balance_type,
                risk_group,
            )

            if key not in rate_lookup:
                rate_lookup[key] = reserve_rate
            else:
                existing_rate = rate_lookup[key]

                if abs(existing_rate - reserve_rate) > EPSILON:
                    conflicts.append({
                        "position": pos,
                        "index": df_out.index[pos],
                        "period": period,
                        "Баланс/внебаланс": balance_type,
                        "ГР": risk_group,
                        "ставка_1": existing_rate,
                        "ставка_2": reserve_rate,
                    })

    rate_lookup_conflicts = pd.DataFrame(conflicts)

    if conflicts:
        raise ValueError(
            "В df_out найдены разные %рез для одинаковых "
            "сочетаний Баланс/внебаланс + ГР. "
            "Исправь ГР/%рез и повтори сохранение."
        )

    return rate_lookup


def get_intermediate_rate(
    risk_group,
    balance_type,
    prev_group,
    prev_rate,
    current_group,
    current_rate,
):
    """
    Возвращает %рез для промежуточной ГР.

    1. Если ГР совпала со старой фактической — старый фактический %рез.
    2. Если ГР совпала с новой фактической — новый фактический %рез.
    3. Иначе ставка берется из актуального rate_lookup, построенного из df_out.
    """
    if same_group(risk_group, prev_group):
        return prev_rate

    if same_group(risk_group, current_group):
        return current_rate

    balance_type = normalize_scalar(balance_type)
    risk_group = to_float(risk_group)

    if (
        balance_type is None
        or pd.isna(risk_group)
    ):
        return np.nan

    return rate_lookup.get(
        (balance_type, risk_group),
        np.nan,
    )



def calculate_ordinary_risk_group(
    ni,
    pfn,
    nvv,
    security,
):
    """
    Определяет ПРОМЕЖУТОЧНУЮ ГР для факторного анализа.

    Это не заменяет фактическую ГР из df_out.
    Функция нужна только для моделирования последовательных переходов,
    когда за один период изменилось несколько признаков.
    """

    security = normalize_scalar(security)

    if security is None:
        return np.nan


    # Высококачественное обеспечение -> 1 ГР.
    if "высококачествен" in security:
        return 1


    # Обеспеченный:
    # ПФН -> 3 ГР
    # НИ / НВВ -> 2 ГР
    # иначе -> 1 ГР
    if security == "обеспеченный":

        if pfn:
            return 3

        if ni or nvv:
            return 2

        return 1


    # Недостаточно обеспеченный:
    # ПФН -> 3 ГР
    # иначе -> 2 ГР
    if (
        "недостаточно" in security
        and
        "обеспеч" in security
    ):

        if pfn:
            return 3

        return 2


    # Не обеспеченный:
    # ПФН -> 4 ГР
    # НИ / НВВ -> 3 ГР
    # иначе -> 2 ГР
    if security in {
        "не обеспеченный",
        "необеспеченный",
    }:

        if pfn:
            return 4

        if ni or nvv:
            return 3

        return 2


    return np.nan


def apply_factor_to_state(
    factor,
    state,
    current_state,
):
    """Применяет к промежуточному состоянию только один изменившийся фактор."""

    new_state = state.copy()

    if factor == "NI":
        new_state["NI"] = current_state["NI"]

    elif factor == "PFN":
        new_state["PFN"] = current_state["PFN"]

    elif factor == "NVV":
        new_state["NVV"] = current_state["NVV"]

    elif factor == "REESTR":
        new_state["REESTR"] = current_state["REESTR"]

    elif factor == "SECURITY":
        new_state["SECURITY"] = current_state["SECURITY"]

    return new_state


def group_after_applying_factor(
    factor,
    state,
    current_state,
):
    """Определяет промежуточную ГР после применения одного фактора."""

    temp_state = apply_factor_to_state(
        factor=factor,
        state=state,
        current_state=current_state,
    )


    # Для состояния в рестре обычная матрица ГР не применяется.
    if temp_state["REESTR"]:
        return np.nan


    return calculate_ordinary_risk_group(
        ni=temp_state["NI"],
        pfn=temp_state["PFN"],
        nvv=temp_state["NVV"],
        security=temp_state["SECURITY"],
    )


def select_next_factor_index(
    pending_factors,
    state_group,
    final_group,
    state,
    current_state,
):
    """
    Выбирает следующий фактор при одновременном изменении нескольких признаков.

    Приоритет:
    1. Фактор, дающий переход ровно на следующую ГР к итоговой.
    2. Фактор, максимально приближающий ГР к итоговой.
    3. Если определить невозможно — исходный порядок факторов.
    """

    state_group = to_float(state_group)
    final_group = to_float(final_group)


    # Если ГР нечисловая, берем первый обычный фактор.
    if (
        pd.isna(state_group)
        or pd.isna(final_group)
    ):

        for i, factor in enumerate(pending_factors):

            if factor != "REESTR":
                return i

        return 0


    # Направление изменения ГР.
    if final_group > state_group:
        direction = 1

    elif final_group < state_group:
        direction = -1

    else:
        return 0


    desired_group = (
        state_group
        +
        direction
    )


    # -------------------------------------------------------------------------
    # 1. Ищем фактор, который дает ровно следующую ГР.
    # -------------------------------------------------------------------------

    for i, factor in enumerate(pending_factors):

        if factor == "REESTR":
            continue


        candidate_group = group_after_applying_factor(
            factor=factor,
            state=state,
            current_state=current_state,
        )

        candidate_group = to_float(candidate_group)


        if pd.isna(candidate_group):
            continue


        if candidate_group == desired_group:
            return i


    # -------------------------------------------------------------------------
    # 2. Ищем фактор, который максимально приближает ГР к фактической.
    # -------------------------------------------------------------------------

    current_distance = abs(
        final_group
        -
        state_group
    )

    best_index = None
    best_distance = np.inf


    for i, factor in enumerate(pending_factors):

        if factor == "REESTR":
            continue


        candidate_group = group_after_applying_factor(
            factor=factor,
            state=state,
            current_state=current_state,
        )

        candidate_group = to_float(candidate_group)


        if pd.isna(candidate_group):
            continue


        distance = abs(
            final_group
            -
            candidate_group
        )


        if (
            distance < current_distance
            and distance < best_distance
        ):

            best_distance = distance
            best_index = i


    if best_index is not None:
        return best_index


    # -------------------------------------------------------------------------
    # 3. Если лучший переход определить нельзя — сохраняем порядок.
    # -------------------------------------------------------------------------

    for i, factor in enumerate(pending_factors):

        if factor != "REESTR":
            return i


    return 0



def add_contribution(
    output,
    factor,
    amount,
    prev_state,
    current_state,
):
    """Записывает рассчитанный эффект в нужный факторный столбец."""

    if factor == "NI":

        if (
            prev_state["NI"]
            and
            not current_state["NI"]
        ):
            output["ушла НИ"] += amount

        else:
            output["пришла НИ"] += amount


    elif factor == "PFN":

        if (
            prev_state["PFN"]
            and
            not current_state["PFN"]
        ):
            output["ушел ПФН"] += amount

        else:
            output["пришел ПФН"] += amount


    elif factor == "NVV":

        if (
            prev_state["NVV"]
            and
            not current_state["NVV"]
        ):
            output["ушло НВВ"] += amount

        else:
            output["пришло НВВ"] += amount


    elif factor == "REESTR":

        output["изменение рестры"] += amount


    elif factor == "SECURITY":

        output["изменилась обеспеченность"] += amount


def add_status(messages, message):
    """Добавляет уникальное сообщение в список замечаний."""

    if message not in messages:
        messages.append(message)


def validate_factor_signs(
    output,
    messages,
):
    """Проверяет ожидаемый знак прихода / ухода негативных факторов."""

    if output["ушла НИ"] < -EPSILON:
        add_status(
            messages,
            "Ушла НИ, но эффект отрицательный",
        )

    if output["пришла НИ"] > EPSILON:
        add_status(
            messages,
            "Пришла НИ, но эффект положительный",
        )

    if output["ушел ПФН"] < -EPSILON:
        add_status(
            messages,
            "Ушел ПФН, но эффект отрицательный",
        )

    if output["пришел ПФН"] > EPSILON:
        add_status(
            messages,
            "Пришел ПФН, но эффект положительный",
        )

    if output["ушло НВВ"] < -EPSILON:
        add_status(
            messages,
            "Ушло НВВ, но эффект отрицательный",
        )

    if output["пришло НВВ"] > EPSILON:
        add_status(
            messages,
            "Пришло НВВ, но эффект положительный",
        )



def calculate_quality_factors_for_row(
    row,
    old_period,
    new_period,
):
    """Рассчитывает факторный анализ между двумя соседними отчетными датами."""

    # -------------------------------------------------------------------------
    # Все числовые результаты по умолчанию равны 0.
    # Поэтому даже при невозможности расчета NaN в расчетные поля не попадет.
    # -------------------------------------------------------------------------

    output = {
        "ушла НИ": 0.0,
        "пришла НИ": 0.0,
        "ушел ПФН": 0.0,
        "пришел ПФН": 0.0,
        "ушло НВВ": 0.0,
        "пришло НВВ": 0.0,
        "изменение рестры": 0.0,
        "изменилась обеспеченность": 0.0,
        "ухудшение качества": 0.0,
        "проверка": 0,
    }

    messages = []


    # =========================================================================
    # OD ТЕКУЩЕГО ПЕРИОДА — БАЗА ДЛЯ ВСЕХ ФАКТОРНЫХ ЭФФЕКТОВ
    # =========================================================================

    od = to_float(
        row[
            make_col(
                SOURCE_COLUMNS["od"],
                new_period,
            )
        ]
    )


    # Фактические ГР.
    prev_group = row[
        make_col(
            SOURCE_COLUMNS["group"],
            old_period,
        )
    ]

    current_group = row[
        make_col(
            SOURCE_COLUMNS["group"],
            new_period,
        )
    ]


    # Фактические проценты резервирования напрямую из df_out.
    prev_rate = to_float(
        row[
            make_col(
                SOURCE_COLUMNS["rate"],
                old_period,
            )
        ]
    )

    current_rate = to_float(
        row[
            make_col(
                SOURCE_COLUMNS["rate"],
                new_period,
            )
        ]
    )


    # =========================================================================
    # ТЕХНИЧЕСКИЕ ПРОВЕРКИ
    # =========================================================================
    #
    # Если что-то неожиданно отсутствует, расчетные значения остаются 0,
    # а причина записывается в "проверка".
    # =========================================================================

    if pd.isna(od):

        output["проверка"] = (
            "Не заполнен OD текущего периода"
        )

        return pd.Series(output)


    if pd.isna(prev_rate):

        output["проверка"] = (
            "Не заполнен %рез предыдущего периода"
        )

        return pd.Series(output)


    if pd.isna(current_rate):

        output["проверка"] = (
            "Не заполнен %рез текущего периода"
        )

        return pd.Series(output)


    # =========================================================================
    # ОБЩЕЕ ИЗМЕНЕНИЕ КАЧЕСТВА
    # =========================================================================

    total_effect = (
        -od
        *
        (
            current_rate
            -
            prev_rate
        )
        /
        100
    )

    output[
        "ухудшение качества"
    ] = total_effect


    # =========================================================================
    # СОСТОЯНИЕ ФАКТОРОВ В ПРЕДЫДУЩЕМ ПЕРИОДЕ
    # =========================================================================

    prev_state = {

        "NI": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["ni"],
                    old_period,
                )
            ]
        ),

        "PFN": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["pfn"],
                    old_period,
                )
            ]
        ),

        "NVV": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["nvv"],
                    old_period,
                )
            ]
        ),

        "REESTR": contains_restra(
            row[
                make_col(
                    SOURCE_COLUMNS["restra"],
                    old_period,
                )
            ]
        ),

        "SECURITY": row[
            make_col(
                SOURCE_COLUMNS["security"],
                old_period,
            )
        ],
    }


    # =========================================================================
    # СОСТОЯНИЕ ФАКТОРОВ В ТЕКУЩЕМ ПЕРИОДЕ
    # =========================================================================

    current_state = {

        "NI": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["ni"],
                    new_period,
                )
            ]
        ),

        "PFN": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["pfn"],
                    new_period,
                )
            ]
        ),

        "NVV": is_active(
            row[
                make_col(
                    SOURCE_COLUMNS["nvv"],
                    new_period,
                )
            ]
        ),

        "REESTR": contains_restra(
            row[
                make_col(
                    SOURCE_COLUMNS["restra"],
                    new_period,
                )
            ]
        ),

        "SECURITY": row[
            make_col(
                SOURCE_COLUMNS["security"],
                new_period,
            )
        ],
    }


    # =========================================================================
    # ИЩЕМ ИЗМЕНИВШИЕСЯ ФАКТОРЫ
    # =========================================================================

    changed_factors = []


    if prev_state["NI"] != current_state["NI"]:
        changed_factors.append("NI")

    if prev_state["PFN"] != current_state["PFN"]:
        changed_factors.append("PFN")

    if prev_state["NVV"] != current_state["NVV"]:
        changed_factors.append("NVV")

    if prev_state["REESTR"] != current_state["REESTR"]:
        changed_factors.append("REESTR")

    if (
        normalize_scalar(prev_state["SECURITY"])
        !=
        normalize_scalar(current_state["SECURITY"])
    ):
        changed_factors.append("SECURITY")


    # =========================================================================
    # ЕСЛИ ОБЩИЙ ЭФФЕКТ РАВЕН 0
    # =========================================================================

    if abs(total_effect) < EPSILON:
        return pd.Series(output)


    # =========================================================================
    # %РЕЗ ИЗМЕНИЛСЯ, НО НИ ОДИН ИЗ ИЗВЕСТНЫХ ФАКТОРОВ НЕ ИЗМЕНИЛСЯ
    # =========================================================================

    if len(changed_factors) == 0:

        output["проверка"] = (
            "Изменился %рез, но изменений "
            "НИ/ПФН/НВВ/рестры/обеспеченности не найдено"
        )

        return pd.Series(output)


    # =========================================================================
    # ИЗМЕНИЛСЯ РОВНО ОДИН ФАКТОР
    # =========================================================================
    #
    # Весь эффект однозначно относится на него.
    # =========================================================================

    if len(changed_factors) == 1:

        factor = changed_factors[0]

        add_contribution(
            output=output,
            factor=factor,
            amount=total_effect,
            prev_state=prev_state,
            current_state=current_state,
        )

        validate_factor_signs(
            output,
            messages,
        )

        output["проверка"] = (
            "; ".join(messages)
            if messages
            else 0
        )

        return pd.Series(output)


    # =========================================================================
    # ИЗМЕНИЛОСЬ ДВА ИЛИ БОЛЕЕ ФАКТОРОВ
    # =========================================================================
    #
    # Используем последовательную логику переходов из исходного VBA.
    # =========================================================================

    pending_factors = []


    # Если договор ВЫШЕЛ из рестры — снимаем рестру первой.
    if (
        prev_state["REESTR"]
        and
        not current_state["REESTR"]
    ):
        pending_factors.append("REESTR")


    # Базовая последовательность обычных факторов.
    if prev_state["NI"] != current_state["NI"]:
        pending_factors.append("NI")

    if prev_state["PFN"] != current_state["PFN"]:
        pending_factors.append("PFN")

    if prev_state["NVV"] != current_state["NVV"]:
        pending_factors.append("NVV")

    if (
        normalize_scalar(prev_state["SECURITY"])
        !=
        normalize_scalar(current_state["SECURITY"])
    ):
        pending_factors.append("SECURITY")


    # Если договор ПРИШЕЛ в рестру — рестра применяется последней.
    if (
        not prev_state["REESTR"]
        and
        current_state["REESTR"]
    ):
        pending_factors.append("REESTR")


    # Начинаем со старого фактического состояния.
    state = prev_state.copy()
    state_group = prev_group
    state_rate = prev_rate

    last_factor = None


    # =========================================================================
    # ПОСЛЕДОВАТЕЛЬНО ПРИМЕНЯЕМ ФАКТОРЫ
    # =========================================================================

    while len(pending_factors) > 0:


        # Если остался один фактор — выбор очевиден.
        if len(pending_factors) == 1:

            selected_index = 0


        # Если промежуточное состояние пока находится в рестре,
        # в первую очередь пытаемся выйти из нее.
        elif state["REESTR"]:

            if "REESTR" in pending_factors:

                selected_index = (
                    pending_factors.index(
                        "REESTR"
                    )
                )

            else:

                selected_index = 0


        else:

            selected_index = (
                select_next_factor_index(
                    pending_factors=pending_factors,
                    state_group=state_group,
                    final_group=current_group,
                    state=state,
                    current_state=current_state,
                )
            )


        factor = pending_factors[
            selected_index
        ]

        last_factor = factor


        # =====================================================================
        # ПОСЛЕДНИЙ ФАКТОР
        # =====================================================================
        #
        # Он всегда доводит расчет до фактической текущей ГР и %рез.
        # Это обеспечивает равенство общей суммы факторному эффекту.
        # =====================================================================

        if len(pending_factors) == 1:

            next_group = current_group
            next_rate = current_rate


        # =====================================================================
        # ПРОМЕЖУТОЧНЫЙ ФАКТОР
        # =====================================================================

        else:

            next_group = (
                group_after_applying_factor(
                    factor=factor,
                    state=state,
                    current_state=current_state,
                )
            )


            # Если промежуточную ГР определить невозможно,
            # этот шаг получает нулевой эффект.
            if pd.isna(
                to_float(next_group)
            ):

                next_group = state_group
                next_rate = state_rate

                add_status(
                    messages,
                    (
                        "Не удалось определить промежуточную "
                        f"ГР после фактора {factor}"
                    ),
                )


            else:

                # Процент промежуточной ГР берем только из df_out.
                next_rate = (
                    get_intermediate_rate(
                        risk_group=next_group,
                        balance_type=row[
                            BALANCE_COLUMN
                        ],
                        prev_group=prev_group,
                        prev_rate=prev_rate,
                        current_group=current_group,
                        current_rate=current_rate,
                    )
                )


                # Если такая промежуточная ГР в df_out не встретилась,
                # числовой эффект этого шага остается нулевым.
                if pd.isna(next_rate):

                    add_status(
                        messages,
                        (
                            "В df_out не найден %рез "
                            f"для промежуточной ГР {next_group}"
                        ),
                    )

                    next_rate = state_rate


        # =====================================================================
        # ЭФФЕКТ ТЕКУЩЕГО ФАКТОРА
        # =====================================================================
        #
        # ВАЖНО: используется OD ТЕКУЩЕГО ПЕРИОДА.
        # =====================================================================

        contribution = (
            od
            *
            (
                state_rate
                -
                next_rate
            )
            /
            100
        )


        add_contribution(
            output=output,
            factor=factor,
            amount=contribution,
            prev_state=prev_state,
            current_state=current_state,
        )


        # Обновляем промежуточное состояние.
        state = apply_factor_to_state(
            factor=factor,
            state=state,
            current_state=current_state,
        )

        state_group = next_group
        state_rate = next_rate


        # Удаляем уже обработанный фактор.
        pending_factors.pop(
            selected_index
        )


    # =========================================================================
    # КОНТРОЛЬНЫЙ ОСТАТОК
    # =========================================================================

    factor_sum = sum(
        output[column]
        for column in FACTOR_COLUMNS
    )

    residual = (
        total_effect
        -
        factor_sum
    )


    # Остаток относим на последний фактор.
    # Это повторяет механизм контрольного выравнивания исходного алгоритма.
    if abs(residual) >= EPSILON:

        add_contribution(
            output=output,
            factor=last_factor,
            amount=residual,
            prev_state=prev_state,
            current_state=current_state,
        )

        add_status(
            messages,
            (
                f"Остаток {residual:.6f} "
                f"отнесен на последний фактор {last_factor}"
            ),
        )


    # =========================================================================
    # ФИНАЛЬНАЯ ПРОВЕРКА
    # =========================================================================

    final_factor_sum = sum(
        output[column]
        for column in FACTOR_COLUMNS
    )


    if (
        abs(
            final_factor_sum
            -
            total_effect
        )
        >=
        EPSILON
    ):

        add_status(
            messages,
            "Сумма факторов не равна ухудшению качества",
        )


    validate_factor_signs(
        output,
        messages,
    )


    output["проверка"] = (
        "; ".join(messages)
        if messages
        else 0
    )


    return pd.Series(output)



def calculate_revaluation(
    df,
    old_period,
    new_period,
):
    """
    Переоценка:

    задолженность прошлого периода
    * (курс текущего периода - курс прошлого периода)
    * %рез прошлого периода / 100
    """

    prev_debt = pd.to_numeric(
        df[make_col(SOURCE_COLUMNS["debt"], old_period)],
        errors="coerce",
    ).fillna(0.0)

    prev_rate = pd.to_numeric(
        df[make_col(SOURCE_COLUMNS["rate"], old_period)],
        errors="coerce",
    ).fillna(0.0)

    current_fx = df[COL_CURRENCY].apply(
        lambda currency: get_fx_rate(currency, new_period)
    )

    prev_fx = df[COL_CURRENCY].apply(
        lambda currency: get_fx_rate(currency, old_period)
    )

    current_fx = pd.to_numeric(
        current_fx,
        errors="coerce",
    ).fillna(0.0)

    prev_fx = pd.to_numeric(
        prev_fx,
        errors="coerce",
    ).fillna(0.0)

    return (
        prev_debt
        * (current_fx - prev_fx)
        * prev_rate
        / 100
    )


def calculate_portfolio_change(
    df,
    old_period,
    new_period,
):
    """
    Изменение портфеля:

    -(OD текущее * %рез прошлое / 100
      - OD прошлое * %рез прошлое / 100)
    - переоценка

    ВАЖНО:
    для обеих величин OD используется %рез ПРЕДЫДУЩЕГО периода.
    """

    current_od = pd.to_numeric(
        df[make_col(SOURCE_COLUMNS["od"], new_period)],
        errors="coerce",
    ).fillna(0.0)

    prev_od = pd.to_numeric(
        df[make_col(SOURCE_COLUMNS["od"], old_period)],
        errors="coerce",
    ).fillna(0.0)

    prev_rate = pd.to_numeric(
        df[make_col(SOURCE_COLUMNS["rate"], old_period)],
        errors="coerce",
    ).fillna(0.0)

    revaluation = calculate_revaluation(
        df,
        old_period,
        new_period,
    )

    return (
        -(
            current_od * prev_rate / 100
            -
            prev_od * prev_rate / 100
        )
        - revaluation
    )


# =============================================================================
# АВТОМАТИЧЕСКИЙ ПЕРЕСЧЕТ РАСХОДОВ / ФАКТОРНОГО АНАЛИЗА
# =============================================================================

def recalculate_reserve_expenses(silent=False):
    """
    Полностью пересчитывает расчетные факторные столбцы по текущему df_out.

    Вызывается:
    - один раз перед открытием редактора;
    - автоматически после каждого сохранения страницы через ipywidgets.

    После каждого сохранения пересчитываются:
    - факторное изменение качества;
    - переоценка;
    - изменение портфеля.
    """
    global PERIODS

    PERIODS = find_periods(df_out)

    if not PERIODS:
        raise ValueError(
            "Не найдены столбцы вида 'задолженность_01.01.2026'."
        )

    # Проверяем обязательные исходные столбцы.
    missing_columns = []

    for period in PERIODS:
        for source_column in SOURCE_COLUMNS.values():
            column_name = make_col(source_column, period)

            if column_name not in df_out.columns:
                missing_columns.append(column_name)

    if BALANCE_COLUMN not in df_out.columns:
        missing_columns.append(BALANCE_COLUMN)

    if missing_columns:
        raise KeyError(
            "В df_out отсутствуют обязательные столбцы:\n"
            + "\n".join(missing_columns)
        )

    # Создаем расчетные столбцы, если их еще нет.
    for period in PERIODS:
        for column in CALC_COLUMNS:
            full_column_name = make_col(column, period)

            if full_column_name not in df_out.columns:
                if column == "проверка":
                    df_out[full_column_name] = 0
                else:
                    df_out[full_column_name] = 0.0

    # После ручных изменений ГР/%рез справочник всегда строится заново.
    rebuild_rate_lookup()

    # Первая отчетная дата: расчетные эффекты равны 0.
    first_period = PERIODS[0]

    for factor in FACTOR_COLUMNS:
        df_out[make_col(factor, first_period)] = 0.0

    df_out[make_col("ухудшение качества", first_period)] = 0.0
    df_out[make_col("проверка", first_period)] = 0
    df_out[make_col("переоценка", first_period)] = 0.0
    df_out[make_col("изменение портфеля", first_period)] = 0.0

    # Все последующие даты.
    for old_period, new_period in zip(
        PERIODS[:-1],
        PERIODS[1:],
    ):
        result = df_out.apply(
            calculate_quality_factors_for_row,
            axis=1,
            old_period=old_period,
            new_period=new_period,
        )

        for column in (
            FACTOR_COLUMNS
            + [
                "ухудшение качества",
                "проверка",
            ]
        ):
            df_out[
                make_col(column, new_period)
            ] = result[column]

        df_out[
            make_col("переоценка", new_period)
        ] = calculate_revaluation(
            df_out,
            old_period,
            new_period,
        )

        df_out[
            make_col("изменение портфеля", new_period)
        ] = calculate_portfolio_change(
            df_out,
            old_period,
            new_period,
        )

    if not silent:
        print(
            f"Расходы на резервы / факторный анализ пересчитаны "
            f"для {len(PERIODS)} отчетных дат и {len(df_out):,} строк."
        )

    return {
        "periods": len(PERIODS),
        "rows": len(df_out),
    }


# Первичный расчет после того, как факторы уже протянуты по датам.
recalculate_reserve_expenses(silent=False)


## 2. Табличный редактор

In [ ]:

# =============================================================================
# ЖУРНАЛ РУЧНЫХ ИЗМЕНЕНИЙ
# =============================================================================

if "manual_edit_log" not in globals():
    manual_edit_log = pd.DataFrame(
        columns=[
            "index",
            "позиция строки",
            "УНП",
            "Номер договора",
            "дата",
            "поле",
            "старое значение",
            "новое значение",
        ]
    )


# =============================================================================
# СЛУЖЕБНЫЕ ФУНКЦИИ
# =============================================================================

def safe_float(value):
    try:
        if pd.isna(value):
            return 0.0
        return float(value)
    except Exception:
        return 0.0


def safe_binary(value):
    try:
        return 1 if float(value) == 1 else 0
    except Exception:
        return 0


def safe_gr(value):
    try:
        value = int(float(value))

        if value in [0, 1, 2, 3, 4, 5, 6]:
            return value

        return 0

    except Exception:
        return 0


def safe_text(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in {
        "",
        "nan",
        "none",
        "0",
        "0.0",
    }:
        return ""

    return value


def ensure_object_column(column):
    if column not in df_out.columns:
        return

    if not pd.api.types.is_object_dtype(
        df_out[column].dtype
    ):
        df_out[column] = df_out[column].astype(
            object
        )



# =============================================================================
# ПРОТЯГИВАНИЕ ФАКТОРОВ ВПЕРЕД ПОСЛЕ РУЧНОГО ИЗМЕНЕНИЯ
# =============================================================================
#
# Если на отчетной дате i пользователь поменял состояние факторов и нажал
# "Сохранить страницу", значения этой даты становятся новым состоянием и
# автоматически переносятся на i+1, i+2, ... до последней отчетной даты.
#
# Задолженность и OD не протягиваются.
# =============================================================================

def propagate_factors_forward(row_position, current_date):

    if current_date not in report_dates:
        return 0

    current_date_index = report_dates.index(current_date)
    future_dates = report_dates[current_date_index + 1:]

    propagated_count = 0

    for factor in FACTOR_NAMES:

        current_col = f"{factor}_{current_date}"

        if current_col not in df_out.columns:
            continue

        current_col_pos = df_out.columns.get_loc(current_col)

        if not isinstance(current_col_pos, (int, np.integer)):
            raise ValueError(
                f"В df_out несколько столбцов с названием '{current_col}'"
            )

        current_value = df_out.iat[
            row_position,
            current_col_pos,
        ]

        for future_date in future_dates:

            future_col = f"{factor}_{future_date}"

            if future_col not in df_out.columns:
                continue

            if factor in {"рестра", "обеспеченность"}:
                ensure_object_column(future_col)

            future_col_pos = df_out.columns.get_loc(future_col)

            if not isinstance(future_col_pos, (int, np.integer)):
                raise ValueError(
                    f"В df_out несколько столбцов с названием '{future_col}'"
                )

            old_future_value = df_out.iat[
                row_position,
                future_col_pos,
            ]

            df_out.iat[
                row_position,
                future_col_pos,
            ] = current_value

            old_norm = (
                ""
                if pd.isna(old_future_value)
                else str(old_future_value).strip()
            )

            new_norm = (
                ""
                if pd.isna(current_value)
                else str(current_value).strip()
            )

            if old_norm != new_norm:
                propagated_count += 1

    return propagated_count


# =============================================================================
# СПИСКИ ФИЛЬТРОВ
# =============================================================================

currency_options = (
    ["Все"]
    +
    sorted(
        df_out[COL_CURRENCY]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
)

operation_options = (
    ["Все"]
    +
    sorted(
        df_out[COL_OPERATION]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
)


# =============================================================================
# ПАНЕЛЬ ПОИСКА И ФИЛЬТРОВ
# =============================================================================

search_unn = widgets.Text(
    placeholder="УНП...",
    description="УНП:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "80px"},
)

search_client = widgets.Text(
    placeholder="Часть названия клиента...",
    description="Клиент:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "80px"},
)

search_contract = widgets.Text(
    placeholder="Номер договора...",
    description="Договор:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "80px"},
)

currency_filter = widgets.Dropdown(
    options=currency_options,
    value="Все",
    description="Валюта:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "80px"},
)

operation_filter = widgets.Dropdown(
    options=operation_options,
    value="Все",
    description="Операция:",
    layout=widgets.Layout(width="360px"),
    style={"description_width": "80px"},
)

date_selector = widgets.Dropdown(
    options=report_dates,
    value=report_dates[0],
    description="Дата:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "80px"},
)

page_size_selector = widgets.Dropdown(
    options=[10, 25, 50],
    value=25,
    description="Строк:",
    layout=widgets.Layout(width="190px"),
    style={"description_width": "70px"},
)

apply_filters_button = widgets.Button(
    description="Применить",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="150px"),
)

reset_filters_button = widgets.Button(
    description="Сбросить",
    icon="refresh",
    layout=widgets.Layout(width="140px"),
)


# =============================================================================
# ПАГИНАЦИЯ
# =============================================================================

prev_button = widgets.Button(
    description="← Назад",
    layout=widgets.Layout(width="120px"),
)

next_button = widgets.Button(
    description="Вперед →",
    layout=widgets.Layout(width="120px"),
)

page_info = widgets.HTML()

result_info = widgets.HTML()


# =============================================================================
# КНОПКА СОХРАНЕНИЯ
# =============================================================================

save_page_button = widgets.Button(
    description="Сохранить страницу в df_out",
    button_style="success",
    icon="save",
    layout=widgets.Layout(
        width="270px",
        height="42px",
    ),
)

reload_page_button = widgets.Button(
    description="Отменить несохраненные",
    icon="undo",
    layout=widgets.Layout(
        width="230px",
        height="42px",
    ),
)

status_output = widgets.Output()


# =============================================================================
# СОСТОЯНИЕ РЕДАКТОРА
# =============================================================================

editor_state = {
    "page": 0,
    "positions": np.arange(
        len(df_out),
        dtype=int
    ),
    "rows": [],
}


# =============================================================================
# ФИЛЬТРАЦИЯ
# =============================================================================

def get_filtered_positions():
    mask = np.ones(
        len(df_out),
        dtype=bool
    )

    unn = search_unn.value.strip().lower()

    if unn:
        mask &= (
            df_out[COL_UNN]
            .astype(str)
            .str.lower()
            .str.contains(
                unn,
                regex=False,
                na=False,
            )
            .to_numpy()
        )

    client = search_client.value.strip().lower()

    if client:
        mask &= (
            df_out[COL_CLIENT]
            .astype(str)
            .str.lower()
            .str.contains(
                client,
                regex=False,
                na=False,
            )
            .to_numpy()
        )

    contract = (
        search_contract.value
        .strip()
        .lower()
    )

    if contract:
        mask &= (
            df_out[COL_CONTRACT]
            .astype(str)
            .str.lower()
            .str.contains(
                contract,
                regex=False,
                na=False,
            )
            .to_numpy()
        )

    if currency_filter.value != "Все":
        mask &= (
            df_out[COL_CURRENCY]
            .astype(str)
            .str.strip()
            .eq(
                str(
                    currency_filter.value
                ).strip()
            )
            .to_numpy()
        )

    if operation_filter.value != "Все":
        mask &= (
            df_out[COL_OPERATION]
            .astype(str)
            .str.strip()
            .eq(
                str(
                    operation_filter.value
                ).strip()
            )
            .to_numpy()
        )

    return np.flatnonzero(
        mask
    )


# =============================================================================
# КОЛОНКИ ДЛЯ ВЫБРАННОЙ ДАТЫ
# =============================================================================

def get_date_columns():
    date = date_selector.value

    return {
        "debt": f"задолженность_{date}",
        "od": f"OD_{date}",
        "ni": f"НИ_{date}",
        "pfn": f"ПФН_{date}",
        "nvv": f"НВВ_{date}",
        "restra": f"рестра_{date}",
        "security": f"обеспеченность_{date}",
        "gr": f"ГР_{date}",
        "reserve": f"%рез_{date}",
    }


def validate_date_columns():
    columns = get_date_columns()

    missing = [
        col
        for col in columns.values()
        if col not in df_out.columns
    ]

    if missing:
        raise ValueError(
            "Для выбранной даты отсутствуют столбцы: "
            + ", ".join(missing)
        )


# =============================================================================
# ЗАГОЛОВКИ ТАБЛИЦЫ
# =============================================================================

HEADER_LAYOUT = [
    ("УНП", "115px"),
    ("Клиент", "260px"),
    ("Договор", "160px"),
    ("Валюта", "90px"),
    ("Операция", "170px"),
    ("Задолженность", "135px"),
    ("OD", "135px"),
    ("НИ", "70px"),
    ("ПФН", "70px"),
    ("НВВ", "70px"),
    ("Рестра", "145px"),
    ("Обеспеченность", "210px"),
    ("ГР", "75px"),
    ("% рез", "95px"),
]

GRID_COLUMNS = " ".join(
    width
    for _, width in HEADER_LAYOUT
)


def static_cell(value, width):
    value = "" if pd.isna(value) else str(value)

    return widgets.HTML(
        value=(
            "<div style='"
            "padding:6px;"
            "white-space:nowrap;"
            "overflow:hidden;"
            "text-overflow:ellipsis;"
            "border-bottom:1px solid #eee;"
            f"width:{width};"
            "'>"
            f"{value}"
            "</div>"
        ),
        layout=widgets.Layout(
            width=width
        ),
    )


def header_cell(title, width):
    return widgets.HTML(
        value=(
            "<div style='"
            "font-weight:600;"
            "padding:6px;"
            "border-bottom:2px solid #999;"
            "white-space:nowrap;"
            "'>"
            f"{title}"
            "</div>"
        ),
        layout=widgets.Layout(
            width=width
        ),
    )


# =============================================================================
# СОЗДАНИЕ ОДНОЙ РЕДАКТИРУЕМОЙ СТРОКИ
# =============================================================================

def create_editor_row(pos):
    date = date_selector.value
    columns = get_date_columns()

    row = df_out.iloc[
        pos
    ]

    currency = row[
        COL_CURRENCY
    ]

    debt_widget = widgets.FloatText(
        value=safe_float(
            row[
                columns["debt"]
            ]
        ),
        layout=widgets.Layout(
            width="135px"
        ),
    )

    od_widget = widgets.FloatText(
        value=safe_float(
            row[
                columns["od"]
            ]
        ),
        disabled=True,
        layout=widgets.Layout(
            width="135px"
        ),
    )

    ni_widget = widgets.Dropdown(
        options=[0, 1],
        value=safe_binary(
            row[
                columns["ni"]
            ]
        ),
        layout=widgets.Layout(
            width="70px"
        ),
    )

    pfn_widget = widgets.Dropdown(
        options=[0, 1],
        value=safe_binary(
            row[
                columns["pfn"]
            ]
        ),
        layout=widgets.Layout(
            width="70px"
        ),
    )

    nvv_widget = widgets.Dropdown(
        options=[0, 1],
        value=safe_binary(
            row[
                columns["nvv"]
            ]
        ),
        layout=widgets.Layout(
            width="70px"
        ),
    )

    restra_widget = widgets.Text(
        value=safe_text(
            row[
                columns["restra"]
            ]
        ),
        layout=widgets.Layout(
            width="145px"
        ),
    )

    security_widget = widgets.Text(
        value=safe_text(
            row[
                columns["security"]
            ]
        ),
        layout=widgets.Layout(
            width="210px"
        ),
    )

    gr_widget = widgets.Dropdown(
        options=[0, 1, 2, 3, 4, 5, 6],
        value=safe_gr(
            row[
                columns["gr"]
            ]
        ),
        layout=widgets.Layout(
            width="75px"
        ),
    )

    reserve_widget = widgets.FloatText(
        value=safe_float(
            row[
                columns["reserve"]
            ]
        ),
        layout=widgets.Layout(
            width="95px"
        ),
    )


    # ---------------------------------------------------------------------
    # OD ПЕРЕСЧИТЫВАЕТСЯ СРАЗУ ПРИ ИЗМЕНЕНИИ ЗАДОЛЖЕННОСТИ
    # ---------------------------------------------------------------------

    rate = get_fx_rate(
        currency,
        date
    )

    def debt_changed(change):
        if change["name"] != "value":
            return

        if pd.isna(rate):
            od_widget.value = 0.0

            with status_output:
                clear_output()
                print(
                    f"Не найден курс для валюты "
                    f"{currency} на {date}"
                )

            return

        od_widget.value = (
            safe_float(
                change["new"]
            )
            *
            rate
        )

    debt_widget.observe(
        debt_changed,
        names="value"
    )


    widgets_for_row = {
        "position": int(pos),
        "debt": debt_widget,
        "od": od_widget,
        "ni": ni_widget,
        "pfn": pfn_widget,
        "nvv": nvv_widget,
        "restra": restra_widget,
        "security": security_widget,
        "gr": gr_widget,
        "reserve": reserve_widget,
        "rate": rate,
    }

    cells = [
        static_cell(
            row[COL_UNN],
            "115px"
        ),
        static_cell(
            row[COL_CLIENT],
            "260px"
        ),
        static_cell(
            row[COL_CONTRACT],
            "160px"
        ),
        static_cell(
            row[COL_CURRENCY],
            "90px"
        ),
        static_cell(
            row[COL_OPERATION],
            "170px"
        ),
        debt_widget,
        od_widget,
        ni_widget,
        pfn_widget,
        nvv_widget,
        restra_widget,
        security_widget,
        gr_widget,
        reserve_widget,
    ]

    return widgets_for_row, cells


# =============================================================================
# ОБЛАСТЬ ТАБЛИЦЫ
# =============================================================================

table_container = widgets.Box(
    layout=widgets.Layout(
        width="100%",
        overflow_x="auto",
        border="1px solid #ddd",
    )
)


# =============================================================================
# ОТРИСОВКА ТЕКУЩЕЙ СТРАНИЦЫ
# =============================================================================

def render_page():
    validate_date_columns()

    positions = editor_state[
        "positions"
    ]

    page_size = int(
        page_size_selector.value
    )

    total_rows = len(
        positions
    )

    total_pages = max(
        1,
        math.ceil(
            total_rows
            /
            page_size
        )
    )

    editor_state["page"] = min(
        max(
            editor_state["page"],
            0
        ),
        total_pages - 1
    )

    page = editor_state[
        "page"
    ]

    start = (
        page
        *
        page_size
    )

    end = min(
        start + page_size,
        total_rows
    )

    current_positions = positions[
        start:end
    ]

    items = []

    for title, width in HEADER_LAYOUT:
        items.append(
            header_cell(
                title,
                width
            )
        )

    editor_rows = []

    for pos in current_positions:
        row_widgets, row_cells = (
            create_editor_row(
                int(pos)
            )
        )

        editor_rows.append(
            row_widgets
        )

        items.extend(
            row_cells
        )

    editor_state["rows"] = (
        editor_rows
    )

    grid = widgets.GridBox(
        children=items,
        layout=widgets.Layout(
            grid_template_columns=GRID_COLUMNS,
            grid_gap="2px 4px",
            align_items="center",
            width="max-content",
        )
    )

    table_container.children = [
        grid
    ]

    result_info.value = (
        f"<b>Найдено строк: {total_rows:,}</b>"
    )

    if total_rows == 0:
        result_info.value += (
            " — по текущим фильтрам ничего не найдено"
        )

    page_info.value = (
        f"<b>Страница "
        f"{page + 1} из {total_pages}</b>"
        f" &nbsp; | &nbsp; "
        f"строки "
        f"{start + 1 if total_rows else 0}"
        f"–{end}"
    )

    prev_button.disabled = (
        page <= 0
    )

    next_button.disabled = (
        page >= total_pages - 1
    )


# =============================================================================
# ПРИМЕНЕНИЕ ФИЛЬТРОВ
# =============================================================================

def apply_filters(button=None):
    editor_state["positions"] = (
        get_filtered_positions()
    )

    editor_state["page"] = 0

    with status_output:
        clear_output()

    render_page()


def reset_filters(button=None):
    search_unn.value = ""
    search_client.value = ""
    search_contract.value = ""

    currency_filter.value = "Все"
    operation_filter.value = "Все"

    editor_state["positions"] = np.arange(
        len(df_out),
        dtype=int
    )

    editor_state["page"] = 0

    render_page()


# =============================================================================
# ПЕРЕКЛЮЧЕНИЕ СТРАНИЦ
# =============================================================================

def previous_page(button=None):
    if editor_state["page"] > 0:
        editor_state["page"] -= 1
        render_page()


def next_page(button=None):
    editor_state["page"] += 1
    render_page()


# =============================================================================
# СОХРАНЕНИЕ ТЕКУЩЕЙ СТРАНИЦЫ
# =============================================================================

def save_current_page(button=None):
    global manual_edit_log

    date = date_selector.value
    columns = get_date_columns()

    # Текстовые столбцы заранее переводим в object,
    # чтобы гарантированно сохранялись новые значения.
    ensure_object_column(
        columns["restra"]
    )

    ensure_object_column(
        columns["security"]
    )

    new_log_rows = []
    missing_rates = []
    propagated_changes = 0

    for row_widgets in editor_state[
        "rows"
    ]:
        pos = row_widgets[
            "position"
        ]

        real_index = df_out.index[
            pos
        ]

        rate = row_widgets[
            "rate"
        ]

        debt_value = safe_float(
            row_widgets[
                "debt"
            ].value
        )

        if pd.isna(rate):
            od_value = 0.0

            missing_rates.append(
                (
                    df_out.iloc[pos][
                        COL_CURRENCY
                    ],
                    date
                )
            )
        else:
            od_value = (
                debt_value
                *
                rate
            )

        # Еще раз синхронизируем отображаемый OD.
        row_widgets[
            "od"
        ].value = (
            od_value
        )

        new_values = {
            columns["debt"]:
                debt_value,

            columns["od"]:
                od_value,

            columns["ni"]:
                row_widgets[
                    "ni"
                ].value,

            columns["pfn"]:
                row_widgets[
                    "pfn"
                ].value,

            columns["nvv"]:
                row_widgets[
                    "nvv"
                ].value,

            columns["restra"]:
                (
                    str(
                        row_widgets[
                            "restra"
                        ].value
                    ).strip()
                    or 0
                ),

            columns["security"]:
                (
                    str(
                        row_widgets[
                            "security"
                        ].value
                    ).strip()
                    or 0
                ),

            columns["gr"]:
                row_widgets[
                    "gr"
                ].value,

            columns["reserve"]:
                row_widgets[
                    "reserve"
                ].value,
        }

        for column, new_value in new_values.items():
            col_pos = df_out.columns.get_loc(
                column
            )

            if not isinstance(
                col_pos,
                (int, np.integer)
            ):
                raise ValueError(
                    f"В df_out несколько столбцов "
                    f"с названием '{column}'"
                )

            old_value = df_out.iat[
                pos,
                col_pos
            ]

            df_out.iat[
                pos,
                col_pos
            ] = new_value

            saved_value = df_out.iat[
                pos,
                col_pos
            ]

            old_normalized = (
                ""
                if pd.isna(old_value)
                else str(old_value).strip()
            )

            new_normalized = (
                ""
                if pd.isna(saved_value)
                else str(saved_value).strip()
            )

            if (
                old_normalized
                !=
                new_normalized
            ):
                field_name = column

                suffix = (
                    f"_{date}"
                )

                if field_name.endswith(
                    suffix
                ):
                    field_name = field_name[
                        :-len(suffix)
                    ]

                new_log_rows.append({
                    "index":
                        real_index,

                    "позиция строки":
                        pos,

                    "УНП":
                        df_out.iloc[pos][
                            COL_UNN
                        ],

                    "Номер договора":
                        df_out.iloc[pos][
                            COL_CONTRACT
                        ],

                    "дата":
                        date,

                    "поле":
                        field_name,

                    "старое значение":
                        old_value,

                    "новое значение":
                        saved_value,
                })

        # После сохранения текущей отчетной даты протягиваем именно факторы
        # этой строки на все последующие отчетные даты.
        propagated_changes += propagate_factors_forward(
            row_position=pos,
            current_date=date,
        )

    if new_log_rows:
        manual_edit_log = pd.concat(
            [
                manual_edit_log,
                pd.DataFrame(
                    new_log_rows
                ),
            ],
            ignore_index=True,
        )

    # После сохранения и протягивания факторов автоматически пересчитываем
    # расходы на резервы / факторный анализ по актуальному df_out.
    reserve_recalc_error = None

    try:
        recalculate_reserve_expenses(
            silent=True
        )
    except Exception as exc:
        # Сами ручные изменения уже сохранены. Ошибку пересчета показываем
        # пользователю, чтобы можно было исправить исходные данные.
        reserve_recalc_error = str(exc)

    with status_output:
        clear_output()

        print(
            f"✓ Сохранено изменений на выбранной дате: "
            f"{len(new_log_rows)}"
        )

        print(
            f"↪ Автоматически протянуто изменений на будущие даты: "
            f"{propagated_changes}"
        )

        if reserve_recalc_error is None:
            print(
                "✓ Расходы на резервы автоматически пересчитаны"
            )
        else:
            print(
                "⚠ Изменения сохранены, но при пересчете расходов "
                "на резервы возникла ошибка:"
            )
            print(reserve_recalc_error)

        if missing_rates:
            unique_missing = sorted(
                set(
                    missing_rates
                )
            )

            print(
                "ВНИМАНИЕ: не найдены курсы:"
            )

            for currency, missing_date in unique_missing:
                print(
                    f"  {currency} — "
                    f"{missing_date}"
                )


# =============================================================================
# СОБЫТИЯ
# =============================================================================

apply_filters_button.on_click(
    apply_filters
)

reset_filters_button.on_click(
    reset_filters
)

prev_button.on_click(
    previous_page
)

next_button.on_click(
    next_page
)

save_page_button.on_click(
    save_current_page
)

reload_page_button.on_click(
    lambda button: render_page()
)

date_selector.observe(
    lambda change: (
        editor_state.update(
            {"page": 0}
        ),
        render_page()
    ),
    names="value",
)

page_size_selector.observe(
    lambda change: (
        editor_state.update(
            {"page": 0}
        ),
        render_page()
    ),
    names="value",
)


# =============================================================================
# ИНТЕРФЕЙС
# =============================================================================

title = widgets.HTML(
    "<h2>Табличный редактор df_out</h2>"
)

help_text = widgets.HTML(
    """
    <div style="margin-bottom:10px;">
    <b>Редактируются:</b>
    задолженность, НИ, ПФН, НВВ, рестра,
    обеспеченность, ГР и % резервирования.
    <br>
    <b>OD</b> пересчитывается автоматически:
    задолженность × курс валюты выбранной даты.
    <br>
    После сохранения <b>НИ, ПФН, НВВ, рестра, обеспеченность, ГР и %рез</b>
    автоматически протягиваются на все последующие отчетные даты.
    <br>
    После этого <b>расходы на резервы / факторный анализ пересчитываются автоматически</b>.
    </div>
    """
)

search_box = widgets.VBox([
    widgets.HBox([
        search_unn,
        search_contract,
    ]),
    search_client,
])

filter_box = widgets.HBox([
    currency_filter,
    operation_filter,
    date_selector,
    page_size_selector,
])

filter_buttons = widgets.HBox([
    apply_filters_button,
    reset_filters_button,
])

pager = widgets.HBox([
    prev_button,
    page_info,
    next_button,
])

save_box = widgets.HBox([
    save_page_button,
    reload_page_button,
])

editor_ui = widgets.VBox([
    title,
    help_text,
    search_box,
    filter_box,
    filter_buttons,
    result_info,
    pager,
    table_container,
    pager,
    save_box,
    status_output,
])

# Первичная загрузка.
editor_state["positions"] = np.arange(
    len(df_out),
    dtype=int
)

render_page()

display(
    editor_ui
)


### Логика факторов во времени

- При первом запуске редактора факторы первой отчетной даты копируются на все последующие даты.
- Если на дате `i` изменить фактор и сохранить страницу, состояние факторов этой строки копируется на даты `i+1`, `i+2`, … до конца.
- `задолженность` и `OD` вперед не протягиваются. `OD` по-прежнему пересчитывается только для выбранной даты из задолженности и курса.


## 3. Журнал ручных изменений

In [ ]:

display(manual_edit_log)


## 4. Проверка текущего `df_out`

In [ ]:

df_out.head()


## 5. Расчетные расходы на резервы

Ниже можно посмотреть рассчитанные факторные эффекты, **изменение качества**, **переоценку** и **изменение портфеля**. Все они автоматически пересчитываются после сохранения изменений через виджет.


### Формула изменения портфеля

Используется исправленная формула:

`-(OD текущего периода × %рез прошлого периода / 100 - OD прошлого периода × %рез прошлого периода / 100) - переоценка`

То есть при расчете движения портфеля **для обоих OD используется ставка резервирования прошлого отчетного периода**.


In [ ]:
calc_preview_cols = [
    col for col in df_out.columns
    if any(
        str(col).startswith(f"{name}_")
        for name in CALC_COLUMNS
    )
]

display(df_out[calc_preview_cols].head())